# Aula 4 — Persistência com banco de dados: SQLite e ORM (exemplos práticos)

Para não versionar arquivos `.db` no repositório do curso, os exemplos
abaixo usam um banco SQLite **em memória** (`:memory:`), que existe só
enquanto o notebook está aberto -- os mesmos comandos funcionam
identicamente com um arquivo real (`sqlite:///tarefas.db`, como em
`main.py`).

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.orm import declarative_base, sessionmaker
from sqlalchemy.pool import StaticPool

Base = declarative_base()

class TarefaDB(Base):
    __tablename__ = "tarefas"
    id = Column(Integer, primary_key=True)
    titulo = Column(String, nullable=False)
    concluida = Column(Boolean, default=False)

engine = create_engine(
    "sqlite:///:memory:",
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
)
Base.metadata.create_all(engine)
SessaoLocal = sessionmaker(bind=engine)

print("Tabelas criadas:", list(Base.metadata.tables.keys()))

## Inserindo e consultando

In [ ]:
sessao = SessaoLocal()

nova_tarefa = TarefaDB(titulo="Estudar SQLAlchemy", concluida=False)
sessao.add(nova_tarefa)
sessao.commit()

todas_tarefas = sessao.query(TarefaDB).all()
for tarefa in todas_tarefas:
    print(tarefa.id, tarefa.titulo, tarefa.concluida)

## Experimento guiado

Adicione mais duas tarefas e rode a consulta de novo -- observe os `id`s sendo gerados automaticamente pelo banco.

## Atualizando e removendo

In [ ]:
tarefa = sessao.query(TarefaDB).filter(TarefaDB.id == 1).first()
tarefa.concluida = True
sessao.commit()

tarefa_atualizada = sessao.query(TarefaDB).filter(TarefaDB.id == 1).first()
print(tarefa_atualizada.titulo, tarefa_atualizada.concluida)

In [ ]:
sessao.delete(tarefa_atualizada)
sessao.commit()

print(sessao.query(TarefaDB).filter(TarefaDB.id == 1).first())  # None

## Integração completa: rodando os testes reais de `main.py` (com FastAPI + SQLAlchemy juntos)

In [ ]:
import subprocess

resultado = subprocess.run(
    ["python3", "-m", "pytest", "-v", "test_main.py"],
    capture_output=True,
    text=True,
)
print(resultado.stdout)

# main.py, ao ser importado pelos testes, cria um arquivo tarefas.db real --
# removemos aqui para não deixar resíduo neste notebook de exemplos.
import os
if os.path.exists("tarefas.db"):
    os.remove("tarefas.db")

## Mini-desafio resolvido

**Desafio:** contar quantas tarefas concluídas existem, usando uma consulta filtrada direto no banco (sem carregar tudo e filtrar em Python).

In [ ]:
sessao.add_all([
    TarefaDB(titulo="Tarefa A", concluida=True),
    TarefaDB(titulo="Tarefa B", concluida=False),
    TarefaDB(titulo="Tarefa C", concluida=True),
])
sessao.commit()

quantidade_concluidas = sessao.query(TarefaDB).filter(TarefaDB.concluida == True).count()
print(f"{quantidade_concluidas} tarefas concluídas.")

## Encerrando a sessão

In [ ]:
sessao.close()
print("Sessão encerrada.")